# Frozen fixed bank with classwise Gaussian Soft-NMS
600 deterministic new LVIS val images. Excludes the known Drive plans, previous validation manifests, original images cache, and COCO development split. If other images were previously evaluated elsewhere, place their filenames in MyDrive/ovd/previously_used_images.txt before starting. Run all cells on an L4. Expect around 30 minutes of detector inference plus image fetching and CPU bootstrap. The progress files allow resuming the same manifest. Report all comparisons; never select another variant using this split.

Primary: bank + Soft-NMS versus unmodified stock five-class prompt; lower two-sided paired 95% bootstrap bound above -0.01 and mean per-request time under 2x. Secondary: fixed chunks; superiority requires lower bound above zero. Matched postprocessing controls are descriptive. Timing covers five requested classes including output construction and suppression, excluding image download and one-time setup. Independent class-scoring AP50 on sampled pairs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install ultralytics open_clip_torch ijson
from pathlib import Path
from collections import defaultdict
from contextlib import ExitStack
from unittest.mock import patch
import sys, tempfile, zipfile, json, time, hashlib
import numpy as np
import torch
from PIL import Image
from transformers.modeling_outputs import BaseModelOutput
from transformers.models.grounding_dino import modeling_grounding_dino as gd

root = Path('/content/drive/MyDrive/ovd')
archive = root / 'bundle_20261124_012119.zip'
tmp = Path(tempfile.mkdtemp())
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if name.startswith('bundle/code/') and name.endswith('.py'):
            path = (tmp / name).resolve()
            assert path.is_relative_to(tmp)
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(z.read(name))
    plan = json.loads(z.read('bundle/plan_100pairs.json'))
    fixed = [json.loads(line) for line in z.read('bundle/fixed_chunk_dets_100pairs.jsonl').splitlines()]
sys.path.insert(0, str(tmp / 'bundle/code'))
from vocab_full import GDino
from full_analysis import APTable
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(), 'Select L4 under Runtime → Change runtime type.'
g = GDino('cuda')
model = g.model
names = plan['names']
index = {c:i for i,c in enumerate(names)}
assert len(index) == len(names) == 1203
import urllib.request, random, itertools, ijson, transformers
from PIL import Image
from vocab_full import fetch_json, LVIS_ANN, clean_lvis, xyxy, h, dev_files, N_DEV
from contextlib import nullcontext
# Optional one filename per line for images used in experiments outside this Drive folder.
extra=root/'previously_used_images.txt'
blocked=set(plan['info'])
for p in root.glob('plan*.json'):
    with p.open('rb') as fh:
        for f,_ in ijson.kvitems(fh,'info'):
            blocked.add(f)
blocked.update(f.name for f in (root/'images').glob('*.jpg'))
for p in root.glob('bank_validation_*/manifest.json'):
    blocked.update(json.loads(p.read_text())['images'])
if extra.exists():
    blocked.update(Path(f.strip()).name for f in extra.read_text().splitlines() if f.strip())
_,ranked=dev_files(str(root))
blocked.update(ranked[:N_DEV])
del _
L=fetch_json(LVIS_ANN,'lvis_v1_val.json',str(root/'lvis_v1_val.json'))
cats=sorted(L['categories'],key=lambda c:c['id'])
assert clean_lvis([c['name'] for c in cats])==names
category=dict(zip([c['id'] for c in cats],names))
gt=defaultdict(list)
for a in L['annotations']:
    gt[a['image_id']].append(dict(cls=category[a['category_id']],box=xyxy(a['bbox'])))
def fname(im): return im['coco_url'].split('/')[-1]
ims=sorted((im for im in L['images'] if im['id'] in gt and im.get('neg_category_ids') and fname(im) not in blocked),
           key=lambda im:hashlib.sha256(('gd-bank-soft-v1/'+fname(im)).encode()).digest())[:600]
assert len(ims)==600
info,pairs={},{}
for im in ims:
    f=fname(im); boxes=gt[im['id']]
    pos=sorted({x['cls'] for x in boxes},key=lambda c:h(f+'|p|'+c))[:3]
    neg=sorted((category[k] for k in im['neg_category_ids']),key=lambda c:h(f+c))[:2]
    info[f]=dict(gt=boxes,present=pos,absent=neg,not_exh=[category[k] for k in im.get('not_exhaustive_category_ids',[])],url=im['coco_url'])
    pairs[f]=pos+neg
chosen=list(info)
plan=dict(names=names,info=info,plan=[dict(image=f,target=c,present=c in info[f]['present']) for f in chosen for c in pairs[f]])
assert not set(chosen)&blocked and all(1<=len(cs)<=5 and len(cs)==len(set(cs)) for cs in pairs.values())
del L,gt
run=root/'fixed_bank_soft_confirm_v1'
run.mkdir(exist_ok=True)
manifest=dict(protocol='soft-confirm-v1',images=chosen,targets=plan['plan'],sigma=.5,threshold=.02,top_k=50,
              bundle_sha256=hashlib.sha256(archive.read_bytes()).hexdigest(),gpu=torch.cuda.get_device_name(),
              transformers=transformers.__version__,torch=str(torch.__version__),checkpoint=getattr(model.config,'_commit_hash',None))
mp=run/'manifest.json'
if mp.exists():
    assert json.loads(mp.read_text())==manifest, 'Existing run has a different manifest or software version.'
else:
    mp.write_text(json.dumps(manifest))
print('Frozen',len(chosen),'images and',len(plan['plan']),'pairs;',len(blocked),'previous filenames excluded.')
cache=run/'images';cache.mkdir(exist_ok=True)
def image(f):
    path=cache/f
    if not path.exists():
        partial=path.with_suffix('.part')
        for attempt in range(4):
            try:
                urllib.request.urlretrieve('https://s3.amazonaws.com/images.cocodataset.org/' + '/'.join(info[f]['url'].rsplit('/', 2)[-2:]), partial)
                with Image.open(partial) as im: im.verify()
                partial.replace(path);break
            except Exception:
                if attempt==3: raise
                time.sleep(2**attempt)
    with Image.open(path) as im: return im.convert('RGB')
def caption(cs):
    return ' . '.join(cs) + ' .'
def inputs(im,cs):
    return g.proc(images=im,text=caption(cs),return_tensors='pt').to(g.dev)
gm = gd.generate_masks_with_special_tokens_and_transfer_map


In [ ]:
@torch.inference_mode()
def cached_text(ids):
    mask,pos=gm(ids)
    bm=mask[:,None] if mask.ndim==3 else mask
    return model.model.text_backbone(ids,bm,torch.zeros_like(ids),pos,
                                     return_dict=True).last_hidden_state

def injected(hidden,mask,pos):
    # All modifications are scoped to one inference and restored on exit.
    ctx=ExitStack()
    try:
        ctx.enter_context(patch.object(gd,'generate_masks_with_special_tokens_and_transfer_map',
                                       return_value=(mask,pos)))
        ctx.enter_context(patch.object(model.model.text_backbone,'forward',
                                       return_value=BaseModelOutput(last_hidden_state=hidden)))
        length=max(256,hidden.shape[1])
        owners=[model.config,model.model.config,*model.modules()]
        visited=set()
        for obj in owners:
            if id(obj) not in visited and isinstance(getattr(obj,'max_text_len',None),int):
                ctx.enter_context(patch.object(obj,'max_text_len',length))
                visited.add(id(obj))
        return ctx
    except BaseException:
        ctx.close()
        raise

small=names[:5]
first=image(sorted(pairs)[0])
x=inputs(first,small)
with torch.inference_mode():
    stock=model(**x)
    h=cached_text(x.input_ids)
    mask,pos=gm(x.input_ids)
    with injected(h,mask,pos):
        ctrl=model(**x)
    torch.testing.assert_close(stock.pred_boxes,ctrl.pred_boxes,atol=1e-5,rtol=1e-5)
    torch.testing.assert_close(stock.logits,ctrl.logits,atol=1e-5,rtol=1e-5)
print('Passed: cached-text control reproduces native boxes and token scores.')

@torch.inference_mode()
def build_bank():
    pieces=[]
    for chunk in g.chunks(names):
        ids=g.tok(caption(chunk),return_tensors='pt')['input_ids'].to(g.dev)
        assert ids.shape[1]<=256
        tokens=cached_text(ids)
        at=1
        for c in chunk:
            sub=g.tokens(c)
            assert ids[0,at:at+len(sub)].tolist()==sub
            pieces.append(tokens[:,at:at+len(sub)].mean(dim=1,keepdim=True))
            at+=len(sub)+1
        assert at==ids.shape[1]-1
    return torch.cat(pieces,dim=1)
bank=build_bank()
n=len(names)
assert bank.shape[1]==n
ids=torch.full((1,n),g.tok.convert_tokens_to_ids('.'),device=g.dev,dtype=torch.long)
identity=torch.eye(n,dtype=torch.bool,device=g.dev)[None]
positions=torch.zeros_like(ids)
ones=torch.ones_like(ids)
zeros=torch.zeros_like(ids)
def bank_call(im):
    v=g.proc.image_processor(images=im,return_tensors='pt').to(g.dev)
    o=model(**v,input_ids=ids,token_type_ids=zeros,attention_mask=ones)
    assert o.logits.shape[-1]>=n and torch.isfinite(o.logits[...,:n]).all()
    return o.pred_boxes[0],o.logits[0,:,:n].sigmoid()

def stock_call(im):
    o=model(**inputs(im,small))
    return o.pred_boxes[0],o.logits[0].sigmoid()

def measure(fn,im):
    g._bb.clear()  # each method pays for its own backbone on each image
    torch.cuda.synchronize()
    start=time.perf_counter()
    result=fn(im)
    torch.cuda.synchronize()
    return result,time.perf_counter()-start

In [ ]:
chunks=g.chunks(names)
chunk_of={c:i for i,ch in enumerate(chunks) for c in ch}
def request5(f):
    cs=pairs[f].copy()
    rng=random.Random('fixed-bank-five/'+f)
    cs+=rng.sample([n for n in names if n not in cs],5-len(cs))
    rng.shuffle(cs)
    return cs

def decode_prompt(im,cs):
    o=model(**inputs(im,cs))
    logits=o.logits[0]
    offset=1
    scores=[]
    for c in cs:
        width=len(g.tokens(c))
        scores.append(logits[:,offset:offset+width].amax(1))
        offset+=width+1
    return o.pred_boxes[0],torch.stack(scores,dim=1).sigmoid()

def report(boxes,scores,cs,request):
    return {c:(boxes,scores[:,cs.index(c)]) for c in request}
def compute(method,im,request):
    if method=='bank':
        boxes,scores=bank_call(im)
        return {c:(boxes,scores[:,index[c]]) for c in request}
    if method=='stock5':
        return report(*decode_prompt(im,request),request,request)
    found={}
    for i in sorted({chunk_of[c] for c in request}):
        ch=chunks[i]
        found.update(report(*decode_prompt(im,ch),ch,[c for c in request if chunk_of[c]==i]))
    return found

import json, numpy as np
def postprocess(dets,variant):
    if not dets or variant=='raw': return dets
    a=np.array(dets,float)[:,:5]
    score=a[:,0].copy()
    b=a[:,1:5]
    ext=np.maximum(0,np.minimum(b[:,None,2:],b[None,:,2:])-np.maximum(b[:,None,:2],b[None,:,:2]))
    area=np.maximum(0,b[:,2:]-b[:,:2]).prod(1)
    inter=ext.prod(2)
    ov=inter/np.maximum(1e-12,area[:,None]+area[None,:]-inter)
    if variant=='vote':
        weight=(ov>=.5)*score[None,:]
        a[:,1:5]=np.where(weight.sum(1)[:,None]>0,(weight@b)/np.maximum(weight.sum(1)[:,None],1e-12),b)
        return a.tolist()
    left=list(range(len(a)));out=[]
    while left:
        best=max(left,key=score.__getitem__)
        left.remove(best)
        if score[best]<.02: break
        out.append([float(score[best]),*b[best].tolist()])
        if variant=='hard':left=[j for j in left if ov[best,j]<=.5]
        elif variant=='soft':score[left]*=np.exp(-ov[best,left]**2/.5)
        else:raise ValueError(variant)
    return out

def class_rows(im, result):
    W,H=im.size
    output={}
    for c,(box,score) in result.items():
        s=score.float().cpu().numpy()
        top=np.argsort(-s)[:50]
        keep=top[s[top]>=.02]
        if not len(keep):
            output[c]=[];continue
        b=box[keep.tolist()]
        xy,wh=b[:,:2],b[:,2:]
        corners=(torch.cat([xy-wh/2,xy+wh/2],1)*b.new_tensor([W,H,W,H])).float().cpu().numpy()
        output[c]=np.column_stack((s[keep],corners)).tolist()
    return output

def suppress(raw):
    return {c:postprocess(d,'soft') for c,d in raw.items()}

orders=list(itertools.permutations(('bank','stock5','fixed')))
parts=run/'per_image';parts.mkdir(exist_ok=True)
with torch.inference_mode():
    for method in orders[0]:
        with (injected(bank,identity,positions) if method=='bank' else nullcontext()):
            g._bb.clear()
            suppress(class_rows(first,compute(method,first,small)))
    with injected(bank,identity,positions):
        c=small[0]
        a=suppress(class_rows(first,compute('bank',first,[c])))[c]
        b=suppress(class_rows(first,compute('bank',first,list(reversed(names[:200])))))[c]
        np.testing.assert_allclose(a,b,atol=1e-5,rtol=1e-5)
    for j,f in enumerate(chosen):
        file=parts/(f+'.json')
        if file.exists(): continue
        im=image(f);req=request5(f);rows=[];timing={}
        for method in orders[j%6]:
            with (injected(bank,identity,positions) if method=='bank' else nullcontext()):
                g._bb.clear();torch.cuda.synchronize();start=time.perf_counter()
                raw=class_rows(im,compute(method,im,req))
                torch.cuda.synchronize();raw_time=time.perf_counter()-start
                corrected=suppress(raw)
                total=time.perf_counter()-start
            timing[method]=raw_time;timing[method+'_soft']=total
            for variant,data in ((method,raw),(method+'_soft',corrected)):
                rows.extend(dict(image=f,target=c,cond=variant,dets=data[c]) for c in pairs[f])
        tmpfile=file.with_suffix('.tmp')
        tmpfile.write_text(json.dumps(dict(rows=rows,timing=timing,request=req),allow_nan=False))
        tmpfile.replace(file)
        if (j+1)%10==0: print(j+1,'/',len(chosen),flush=True)


In [ ]:
records=[];times=[]
for f in chosen:
    d=json.loads((parts/(f+'.json')).read_text())
    records.extend(d['rows']);times.append(d['timing'])
table=APTable(records,plan,'indep')
methods=list(times[0])
ap={m:table.map(m) for m in methods}
lat={m:float(np.mean([t[m] for t in times])) for m in methods}
comparators=['stock5','fixed','stock5_soft','fixed_soft']
rng=np.random.default_rng(20261124)
boot={m:[] for m in comparators}
for i in range(1000):
    w=np.bincount(rng.integers(len(chosen),size=len(chosen)),minlength=len(chosen))
    value=table.map('bank_soft',w)
    for m in comparators: boot[m].append(value-table.map(m,w))
contrasts={m:dict(delta=ap['bank_soft']-ap[m],CI95=np.nanpercentile(boot[m],[2.5,97.5]).tolist()) for m in comparators}
ratio=lat['bank_soft']/lat['stock5']
summary=dict(images=len(chosen),pairs=len(plan['plan']),mAP50=ap,seconds=lat,bank_soft_minus=contrasts,
             latency_ratio=ratio,primary_pass=bool(ratio<2 and contrasts['stock5']['CI95'][0]>-.01),
             secondary_fixed_superiority=bool(contrasts['fixed']['CI95'][0]>0),bootstrap_unit='image')
(run/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
print(json.dumps(summary,indent=2))
print(run/'summary.json')
